In [1]:
from core.blob_storage import fetch_transcription_blob_data
import os
import json
from tqdm import tqdm
from urllib.parse import urlparse
from core.transcription_results_parser import parse_results
from utils.path import solve_path, create_folder_if_not_exists
import pandas as pd

from config import DATA_FOLDER

In [2]:
RESULTS_FOLDER = solve_path('transcription_results', DATA_FOLDER)
RESULTS_FOLDER = create_folder_if_not_exists(RESULTS_FOLDER)
RESULTS_FOLDER

'/home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results'

In [3]:
fetch_transcription_blob_data.blob_list[:3]

['1664b39d-8729-48c1-a85b-8d7706caff24/1664b39d-8729-48c1-a85b-8d7706caff24_report.json',
 '1664b39d-8729-48c1-a85b-8d7706caff24/contenturl_0.json',
 '89bfcd91-a15e-4e01-a4f6-05e3981efcbb/89bfcd91-a15e-4e01-a4f6-05e3981efcbb_report.json']

In [4]:
transcription_bucket = 'f27e9462-e88f-417a-96f8-17aa3908f2dc'

In [5]:
transcription_files = fetch_transcription_blob_data.find_blob(transcription_bucket)

In [6]:
result_files = []
for blob in tqdm(transcription_files):
    file_name = os.path.basename(blob)
    file_path = solve_path(file_name, RESULTS_FOLDER)
    if os.path.exists(file_path):
        print(f"File {file_path} already exists, skipping download.")
        result_files.append(file_path)
        continue
    saved_file = fetch_transcription_blob_data(blob, file_path, as_json=True)
    result_files.append(saved_file)

100%|██████████| 37/37 [00:00<00:00, 59074.70it/s]

File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_0.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_1.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_10.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_11.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_12.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_13.json already exists, skipping download.
File /home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_14.json already exists, skipping download.
File /home/h-pgy/projects/tra

In [7]:
result_files[:3]

['/home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_0.json',
 '/home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_1.json',
 '/home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/contenturl_10.json']

In [8]:
def load_json(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [9]:
parse_results(load_json(result_files[0])).head()

,audiencia,file_name,duration_audiencia,frase,confidence,channel,offset,duration
0,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,boa noite a todas e todas sou um homem branco ...,0.802756,1,40,27160
1,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,que é a maior cidade do brasil que tem mais re...,0.820823,1,27560,27880
2,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,compostagem mas também pode gerar biogás pros ...,0.894802,1,55840,26120
3,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,fala que tem que ser a a contratação dos das c...,0.819671,0,82400,29040
4,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,é eu tô em são paulo mas se eu atravessar o ou...,0.823547,1,112280,29110


In [10]:
result_files[-1]

'/home/h-pgy/projects/transcricao_audiencias_pdm/temp_data/transcription_results/f27e9462-e88f-417a-96f8-17aa3908f2dc_report.json'

In [11]:
result_files = [file for file in result_files if not file.endswith('report.json')]    

In [12]:
dfs = []
for file in result_files:
    json_data = load_json(file)
    df = parse_results(json_data)
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df.head()

,audiencia,file_name,duration_audiencia,frase,confidence,channel,offset,duration
0,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,boa noite a todas e todas sou um homem branco ...,0.802756,1,40,27160
1,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,que é a maior cidade do brasil que tem mais re...,0.820823,1,27560,27880
2,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,compostagem mas também pode gerar biogás pros ...,0.894802,1,55840,26120
3,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,fala que tem que ser a a contratação dos das c...,0.819671,0,82400,29040
4,audiencia_publica_geral,audiencia_publica_geral_clip.wav,67.383167,é eu tô em são paulo mas se eu atravessar o ou...,0.823547,1,112280,29110


In [13]:
df.shape

(3858, 8)

In [14]:
from config import DATA_FOLDER
from utils.path import solve_path

In [15]:
fname = solve_path('dados_final.csv', DATA_FOLDER)

In [16]:
df.to_csv(fname, sep=';', encoding='utf-8', index=False)